In [ ]:
import sys, pathlib
SRC = pathlib.Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from paths import RAW_DATA, INTERIM_DATA, CLEAN_DATA


In [ ]:
import pandas as pd
from pandas.tseries.offsets import MonthEnd
import os

# Morningstar Direct

Authorization Token: Valid up to 24 hrs, copy token from [Analytics Lab](https://analyticslab.morningstar.com), from ribben bar "Analytics Lab" and then copy Authorization Token.  
Rewrite the file: echo 'string' >md_token

In [ ]:
import morningstar_data as md
from morningstar_data.direct import InvestmentIdentifier, Frequency

In [ ]:
token_path = pathlib.Path.home() / '.md_token'  # external, not committed

with open(os.path.expanduser(token_path), "r") as f:
    Token = f.read().strip()

os.environ["MD_AUTH_TOKEN"] = Token

## Importing Factors data (Downloaded Directly from MD)

In [ ]:
df_factors = pd.read_excel(str(RAW_DATA / 'sp500_factors.xlsx'))

print('df_factors shape before dropping nan:', df_factors.shape)

# dropping companies with nan values
df_factors = df_factors.replace(["—", "-", "", " "], pd.NA)
df_factors = df_factors.dropna()

print('df_factors shape after  dropping nan:', df_factors.shape)
display(df_factors.head(2), df_factors.isna().sum(), (df_factors == 0).sum())

## Getting  S&P500 monthly return data 
for all 503 companies

In [ ]:
start_date = '2020-01-31'
end_date   = '2024-12-31'

In [ ]:
sec_id = df_factors['SecId'].tolist()

sp500_monthly_return = md.direct.get_returns(
    investments = sec_id,
    start_date = start_date,
    end_date = end_date,
    freq = Frequency.monthly,
    currency = "USD",
)

merged = (
    sp500_monthly_return
        .rename(columns={"Id": "SecId", "Date": "date"})
        .merge(df_factors[["SecId", "Ticker"]], on="SecId", how="left")
)

sp500_monthly_return = merged[["Ticker", "date", "Monthly Return"]]

display(sp500_monthly_return.head(2))
print(
    'return data shape:', sp500_monthly_return.shape, '\n',
    'number of companies:', len(sec_id), '\n',
    'number of months:', sp500_monthly_return.shape[0] / len(sec_id)
)

sp500_monthly_return.to_csv(str(INTERIM_DATA / '5y_sp_ret.csv'), index=False)

# WDRS

In [ ]:
import wrds
db = wrds.Connection(wrds_username=os.getenv('WRDS_USERNAME'), wrds_password=os.getenv('WRDS_PASSWORD'))

## Getting S&P500 benchmark from CRSP

from 2020-01-31 to 2024-12-31

In [ ]:
start_date = '2020-01-31'
end_date   = '2024-12-31'

In [ ]:
sp500 = db.get_table(library='crsp', table='msi', columns=['date', 'sprtrn'])

sp500['date'] = pd.to_datetime(sp500['date'])
sp500.set_index('date', inplace=True)

df_benchmark = sp500.loc[start_date:end_date]

display(df_benchmark.head(2), df_benchmark.shape)

df_benchmark.to_csv(str(INTERIM_DATA / '5y_sp_bmk.csv'))

## Getting risk-free rate from ff

In [ ]:
ff_data = db.get_table(library='ff', table='factors_monthly', columns=['date', 'rf'])

ff_data['date'] = pd.to_datetime(ff_data['date'])
ff_data['date'] = ff_data['date'] + MonthEnd(0)
ff_data.set_index('date', inplace=True)

rf = ff_data.loc[start_date:end_date]

display(rf.head(2), rf.shape)

rf.to_csv(str(INTERIM_DATA / '5y_rf.csv'))

# Merging data
value factors, risk-free, and benchmark return data

In [ ]:
df_sp500_ret   = pd.read_csv(str(INTERIM_DATA / '5y_sp_ret.csv'))
df_bench = pd.read_csv(str(INTERIM_DATA / '5y_sp_bmk.csv'))

df_sp500_ret['date'] = pd.to_datetime(df_sp500_ret['date'])
df_bench['date'] = pd.to_datetime(df_bench['date'])

df_sp500_ret['date'] = pd.to_datetime(df_sp500_ret['date']) + MonthEnd(0)
df_bench['date']     = pd.to_datetime(df_bench['date'])     + MonthEnd(0)


# table pivot
df_sp500_ret = (
    df_sp500_ret
    .pivot(index='date', columns='Ticker', values='Monthly Return')
    .astype(float)
    .div(100)    
)

# equaly weight return
df_sp500_ret['average ret'] = df_sp500_ret.mean(axis=1)

# merging benchmark 
df_merge = df_sp500_ret.merge(df_bench, on='date', how='inner')

# merging risk-free rate
df_merge = df_merge.merge(rf, on='date', how='inner')

df_merge['excess ret'] = df_merge['average ret'] - df_merge['rf']


df_merge = df_merge.sort_values('date').set_index('date')

display(df_merge.shape)

nan_cols_any = df_merge.columns[df_merge.isna().any()]
print("companies with at at least one Nan:", '\n', nan_cols_any.tolist())


# Final and Cleaned Data

Dropping companies with nan return values

In [ ]:
df_merge = df_merge.dropna(axis=1, how='any')

nan_cols_any = df_merge.columns[df_merge.isna().any()]
print("\ncompanies with at at least one Nan:", nan_cols_any.tolist())

display(df_merge.head(), df_merge.shape)
df_merge.to_csv(str(CLEAN_DATA / '5y_merged.csv'))

In [ ]:
df_factors = df_factors[df_factors['Ticker'].isin(df_merge.columns)].copy()
df_factors = df_factors.reset_index(drop=True) 
df_factors.to_csv(str(INTERIM_DATA / '5y_factors.csv'), index=False)

display(df_factors.head(), df_factors.shape)